# VORQ · 03 — API Server

Starts the FastAPI backend via `uvicorn` in a background thread using `nest_asyncio`.
Run this notebook **before** the Streamlit UI (`04_ui_launcher.ipynb`).

Health check: http://127.0.0.1:8000/health
Interactive docs: http://127.0.0.1:8000/docs

In [ ]:
# Install if needed
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','fastapi','uvicorn[standard]','nest_asyncio','pydantic','requests','-q'])

In [ ]:
import sys, os, json, time, threading, asyncio, importlib.util
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()

# Add repo root to path so engine modules resolve
REPO_ROOT = str(Path(os.getcwd()).parent)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f'REPO_ROOT: {REPO_ROOT}')

In [ ]:
import time as _time
from typing import Optional, List, Dict, Any
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from vorq.engine.orchestrator import SimulationOrchestrator
from vorq.engine.exposure_graph import ExposureGraph

# ── Schemas ──────────────────────────────────────────────────────────────────
class SimulationRequest(BaseModel):
    scenario_text:     str           = Field(..., min_length=3, max_length=2000)
    horizon_days:      Optional[int]   = Field(None, ge=1, le=3650)
    severity_override: Optional[float] = Field(None, ge=0.0, le=1.0)
    mc_iterations:     Optional[int]   = Field(None, ge=100, le=5000)

class QuickScoreRequest(BaseModel):
    scenario_text: str = Field(..., min_length=3, max_length=2000)

class SimulationResponse(BaseModel):
    status:      str
    event:       Dict[str, Any]
    simulation:  Dict[str, Any]
    explanation: Dict[str, Any]

class GraphResponse(BaseModel):
    nodes: List[Dict[str, Any]]
    edges: List[Dict[str, Any]]

class HealthResponse(BaseModel):
    status:      str
    version:     str
    graph_nodes: int
    graph_edges: int

# ── App ──────────────────────────────────────────────────────────────────────
app = FastAPI(title='VORQ — Global Risk Map AI', version='1.0.0')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

_orchestrator: Optional[SimulationOrchestrator] = None
_graph:        Optional[ExposureGraph]          = None
_event_store:  list = []

@app.on_event('startup')
async def startup():
    global _orchestrator, _graph
    _orchestrator = SimulationOrchestrator(mc_iterations=1000)
    _graph        = ExposureGraph()
    print('VORQ API ready ✓')

@app.post('/simulate', response_model=SimulationResponse)
async def simulate(req: SimulationRequest):
    try:
        t0  = _time.time()
        res = _orchestrator.run(scenario_text=req.scenario_text, horizon_days=req.horizon_days, severity_override=req.severity_override, mc_iterations=req.mc_iterations)
        res['_meta'] = {'elapsed_seconds': round(_time.time()-t0, 3)}
        _event_store.append({'event': res['event'], 'risk_score': res['simulation']['overall_risk_score'], 'timestamp': _time.time()})
        if len(_event_store) > 100: _event_store.pop(0)
        return res
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Simulation error: {e}')

@app.post('/quick-score')
async def quick_score(req: QuickScoreRequest):
    try:    return _orchestrator.quick_score(req.scenario_text)
    except Exception as e: raise HTTPException(status_code=500, detail=str(e))

@app.get('/events')
async def list_events(limit: int = 50):
    return {'count': len(_event_store), 'events': _event_store[-limit:]}

@app.get('/graph', response_model=GraphResponse)
async def get_graph(): return _graph.to_json()

@app.get('/graph/summary')
async def graph_summary(): return _graph.summary()

@app.get('/health', response_model=HealthResponse)
async def health():
    s = _graph.summary() if _graph else {'nodes': 0, 'edges': 0}
    return {'status': 'healthy', 'version': '1.0.0', 'graph_nodes': s['nodes'], 'graph_edges': s['edges']}

print('FastAPI app defined ✓')

In [ ]:
import uvicorn

config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning')
server = uvicorn.Server(config)

def _run():
    asyncio.run(server.serve())

t = threading.Thread(target=_run, daemon=True)
t.start()

time.sleep(2.5)  # wait for startup
print('API server started on http://127.0.0.1:8000')

In [ ]:
import requests

# Health check
h = requests.get('http://127.0.0.1:8000/health')
print('Health:', h.json())

# Quick simulation test
r = requests.post('http://127.0.0.1:8000/simulate', json={'scenario_text': 'War between India and China in 1 year', 'mc_iterations': 500})
sim = r.json()['simulation']
print(f'Risk: {sim["overall_risk_score"]} ({sim["risk_level"]})')
print(f'Top companies: {[c["name"] for c in sim["company_impacts"][:3]]}')